# FREUID baseline — full-data training (IDE -> Colab A100 remote kernel)

Trains on the **full dataset (~69k train images)**: train -> infer -> submit.
Compute runs on a Colab Pro A100; output shows in your IDE.

> **Resume support:** if the session disconnects mid-training, re-run the Colab cells and re-run the
> train cell — it automatically resumes from the last completed epoch (saved to Drive after every epoch).

---

## STEP 1 — on Colab (browser; bring up the tunnel)

New Colab notebook -> Runtime type **A100 GPU** (requires Colab Pro) -> add 🔑 Secrets
`NGROK_AUTH_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`. Paste the cells below into Colab and run in order.

```python
# [Colab 1] check GPU + clone + install
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU -> set Runtime to A100')
!pip install -q pyngrok
!git clone --depth 1 https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git
%cd kaggle_freuid_2026
!pip install -q -e .
```

```python
# [Colab 2] Kaggle auth (use Secrets KAGGLE_USERNAME / KAGGLE_KEY — no hardcoded tokens)
import os, json
from google.colab import userdata
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json written')
# ⚠️ Accept the competition rules ('Join Competition') on the comp page first, or downloads 403
```

```python
# [Colab 3] download + unzip the real competition data (creates data/train/train, public_test/...)
!kaggle competitions download -c the-freuid-challenge-2026-ijcai-ecai -p data
!cd data && unzip -q -o '*.zip' && rm -f *.zip
!echo 'train imgs:' && ls data/train/train | wc -l && ls data
```

```python
# [Colab 4] (required) mount Google Drive -> checkpoints/, submissions/ saves auto-land in Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/freuid/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/freuid/submissions', exist_ok=True)
!rm -rf checkpoints submissions
!ln -s /content/drive/MyDrive/freuid/checkpoints checkpoints
!ln -s /content/drive/MyDrive/freuid/submissions submissions
print('checkpoints/, submissions/ -> linked to Drive (saves now go to MyDrive/freuid/)')
```

```python
# [Colab 5] GPU kernel + ngrok tunnel -> print URL
import subprocess, time, socket
from google.colab import userdata
from pyngrok import conf, ngrok
conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN').strip()
JUPYTER_TOKEN = 'freuid'
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
ngrok.kill()
log = open('/tmp/jl.log', 'w')
subprocess.Popen(['jupyter','server','--ip=0.0.0.0','--port=8888','--no-browser','--allow-root',
    f'--ServerApp.token={JUPYTER_TOKEN}','--ServerApp.password=','--ServerApp.allow_origin=*',
    '--ServerApp.disable_check_xsrf=True','--ServerApp.allow_remote_access=True',
    '--ServerApp.root_dir=/content/kaggle_freuid_2026'], stdout=log, stderr=subprocess.STDOUT)
for i in range(30):
    time.sleep(1)
    s = socket.socket()
    if s.connect_ex(('127.0.0.1', 8888)) == 0: s.close(); break
    s.close()
print(ngrok.connect(8888, 'http').public_url + '/?token=' + JUPYTER_TOKEN)
```

-> Copy the printed URL. Keep this Colab tab open — closing it ends the session.

---

## STEP 2 — in my IDE (this notebook)
1. Top-right **kernel picker** -> `Select Another Kernel...` -> `Existing Jupyter Server...` -> paste the URL -> `Python 3`
2. Run the cells below in order — all run on the Colab A100

In [4]:
# Connection check: are we on the Colab GPU + is Drive linked?
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (<- runtime is not GPU!)')
print('cwd:', os.getcwd())
print('checkpoints -> Drive?', os.path.realpath('checkpoints'))  # /content/drive/... means auto-save is on
!ls data && echo 'train imgs:' && ls data/train/train | wc -l

device: cuda
cwd: /content/kaggle_freuid_2026
checkpoints -> Drive? /content/drive/MyDrive/freuid/checkpoints
public_test	       train		 train_sample
sample_submission.csv  train_labels.csv  train_sample_labels.csv
train imgs:
69352


In [ ]:
%%writefile configs/a100_vit_all_strategies.yaml
name: a100_vit_all_strategies
seed: 42
data_dir: data
image_size: 384
val_strategy: cross_domain
val_domain_fraction: 0.2
backbone: vit_base_patch16_384.orig_in21k
pretrained: true
epochs: 20
batch_size: 16
lr: 0.0003
weight_decay: 0.0001
num_workers: 4
label_smoothing: 0.1

In [ ]:
CONFIG = 'configs/a100_vit_all_strategies.yaml'
!python -m freuid.train --config {CONFIG}

In [ ]:
CKPT = 'checkpoints/a100_vit_all_strategies.pt'
OUT = 'submissions/a100_vit_all_strategies.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT} --tta 5
!head -3 {OUT} && wc -l {OUT}

In [ ]:
OUT = 'submissions/a100_vit_all_strategies.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'vit cross-domain val + label smoothing + TTA'

## STEP 3 — (optional) auto-receive results into my local repo too

STEP 1·2 above are the **Colab/Drive side**. Here we set up the **local side** so the results Colab saved to
Drive (`checkpoints/*.pt`) **show up automatically inside this repo folder on my Mac**.

> ⚠️ This is not a repo code change — it is **a few shell commands each person runs once on their own Mac**.
> Teammates run the same with their own Mac + Drive path.
> (The `checkpoints` ignore rule is already in the shared `.gitignore`, so no git config is needed.)

**1. Install Google Drive desktop + sign in** (in the Mac terminal)
```bash
brew install --cask google-drive      # install (Mac admin password once)
# Open the app -> sign in with the SAME Google account you use for Colab
# Settings (gear) -> prefer "Mirror files" (files actually land on local disk)
```

**2. Make the freuid folder in Drive and symlink the repo's `checkpoints` to it**
```bash
# <email> is your account. Mind the localized My Drive folder name (en: "My Drive", ko: "내 드라이브")
DRIVE="$HOME/Library/CloudStorage/GoogleDrive-<email>/My Drive/freuid"
mkdir -p "$DRIVE/checkpoints"

cd <this-repo-path>          # e.g. ~/kaggle_freuid_2026
rmdir checkpoints 2>/dev/null   # remove the existing empty checkpoints/ (only if empty)
ln -s "$DRIVE/checkpoints" checkpoints   # checkpoints -> Drive folder
```

`checkpoints` is already ignored by the shared `.gitignore`, so **this symlink does not show up in `git status`** (no extra setup).

**Verify**
```bash
realpath checkpoints   # OK if it points to /Users/<me>/Library/CloudStorage/.../freuid/checkpoints
git status --short     # nothing checkpoints-related should appear
```

-> From now on: **Colab training -> Drive -> Mac sync -> `checkpoints/baseline_colab.pt` appears in this repo.**

### How to get submissions into the repo?
`submissions/` has a git-tracked `.gitkeep` (keeps the empty folder, shared by all contributors). Turning it
into a Drive symlink like checkpoints would drop that `.gitkeep` and **change the shared repo**, so we don't.
Instead, to see the submission csv inside the repo's `submissions/`, use the copy helper (auto-detects the
Drive path; copies are gitignored, never committed):

```bash
bash scripts/pull_submissions.sh
```

(The csv also stays in the Drive folder `MyDrive/freuid/submissions/` and in the Kaggle submission history.)